In [ ]:
import os

# Uremote_edite you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../../config"

# Collections

The Plex service implements the standard collection interfaces that are common across all services.

## Library Collection

The {py:class}`PlexLibrarySectionCollection <plistsync.services.plex.PlexLibrarySectionCollection>` represents content specific to one of your Plex Libraries. (Plex uses different naming: in the frontend they refer to them as "Libraries", in the backend they are "Sections".)

In [ ]:
from plistsync.services.plex import PlexLibrarySectionCollection

library = PlexLibrarySectionCollection(section_name_or_id="Music")

### Iterating Tracks

The Plex Library collection implements the {py:class}`TrackStream <plistsync.core.collection.TrackStream>` protocol, so you can simply iterate over its tracks.

In [ ]:
# Iterate over tracks
for count, track in enumerate(library.tracks):
    print(f"{track.id} -> {track.title} by {track.artists}")
    if count == 4:
        break

This can be a bit slow depending on the number of tracks you want to iterate. To speed things up you can consider preloading all tracks.

In [ ]:
# ~ 6sec for 20k tracks
library.preload()

### Track Lookup

The Plex library collections implements the {py:class}`LocalLookup <plistsync.core.collection.LocalLookup>` and {py:class}`GlobalLookup <plistsync.core.collection.GlobalLookup>` protocols, allowing you to search for tracks across your Plex server. Specifically, plex supports lookups by `plex_id` and `isrc`.

:::{note}
`plex_id` is Plex’s `ratingKey`. It is only guaranteed to be unique within a single Plex Media Server. If you connect to multiple servers, store it together with a server identifier (e.g. base URL or machine identifier) to avoid collisions.
:::

In [ ]:
# By plex id
if plex_track := library.find_by_local_ids({"plex_id": "14605"}):
    print(plex_track)

Looking up tracks by `isrc` is currently a bit more involved.

Plex does not expose `isrc` codes via its API/metadata, so plistsync can’t retrieve them directly from the server. Instead, plistsync falls back to reading the `isrc` from the audio file’s embedded tags. This requires filesystem access to the underlying media files and can be noticeably slower especially for large libraries or when reading files over a network mount.

In the example below, plistsync is running on a Mac that is not the Plex server. The music library is mounted locally at `/Volumes/music`, while the Plex server itself stores (and therefore reports) track paths under `/media/music`. To reconcile these two path “views”, we translate Plex’s remote paths into local mount paths using a {py:class}`PathRewrite <plistsync.core.PathRewrite>.

In [ ]:
from pathlib import Path

from plistsync.core import PathRewrite

# We create a path rewrite from `/media/music` to `/Volumes/music`
path_rewrite = PathRewrite.from_str("/media/music/", "/Volumes/music/")
local_path = path_rewrite.apply(Path("/media/music/foo/music.mp3"))
print(f"{local_path=}")
print(f"{local_path.is_file()=}")

In [ ]:
# by isrc
if plex_track := library.find_by_global_ids(
    {"isrc": "GBXJH1000082"},
    path_rewrite=path_rewrite,
):
    print(plex_track)

:::{note}
The lookup by isrc can take a bit of time this will read the metadata of all your music files until the isrc is found.

It is possible to speedup the operation using a cache but we will leave
this here for simplicity.
:::

## Playlist Collection

The {py:class}`PlexPlaylistCollection <plistsync.services.plex.PlexPlaylistCollection>` represents a playlist that is accessible by the currently authenticated user.  

### Retrieving playlists

You can retrieve all playlists of a user using the library's {py:meth}`PlexLibrarySectionCollection.playlists <plistsync.services.plex.PlexLibrarySectionCollection.playlists>` property.

In [ ]:
for count, pl in enumerate(library.playlists):
    print(f"{pl.id:7d} {pl.name} ({len(pl)} tracks)")
    if count == 4:
        break

If you just want a specific playlist, you can use the library's {py:meth}`PlexLibraryCollection.get_playlist <plistsync.services.plex.PlexLibraryCollection.get_playlist>` method to retrieve a playlist.

In [ ]:
# Can get via namer or id
pl = library.get_playlist(
    name="DnB Classics"
    # id = 108530
)
if pl is not None:
    print(f"Found playlist: {pl.id} {pl.name} ({len(pl.tracks)} tracks)")
else:
    print("Playlist not found.")

:::{note}
This method supports lookup by various identifiers, currently ``name=`` and ``id=``.
Lookup by name will return ``None`` if no matching playlist is found, while lookups by other identifiers will raise a ``ValueError`` if the playlist cannot be resolved.
:::

### Creating playlists

Use {py.class}`PlexPlaylistCollection <plistsync.services.plex.PlexPlaylistCollection>` to model a playlist locally, then (optionally) create/associate it on Plex (online).

In [ ]:
from plistsync.services.plex import PlexPlaylistCollection

try:
    library.get_playlist(name="My New Playlist").remote_delete()
except:
    print("Playlist did not exist, no need to delete")

pl = PlexPlaylistCollection(
    library=library,
    name="My New Playlist",
    description="Created via plistsync",
)

# currently, server does not know of this playlist.
assert pl.remote_associated == False

To remotly create the same playlist use the {py:meth}`PlexPlaylistCollection.remote_create <plistsync.services.plex.PlexPlaylistCollection.remote_create>` method.

In [ ]:
pl.remote_create()
assert pl.remote_associated == True

### Updating a playlist

For updating a playlist, you should use the playlist's {py:meth}`PlexPlaylistCollection.remote_edit <plistsync.services.plex.PlexPlaylistCollection.remote_edit>` context manager. This ensures that all changes are properly saved back to the server when you exit the context. This also minifies changes and therefore reduces API calls.


In [ ]:
# Updating a playlist description/name
with pl.remote_edit():
    pl.description = "Created via plistsync"

pl.description

In [ ]:
from plistsync.services.plex import PlexTrack

# find some tracks to play around with
new_tracks: list[PlexTrack] = list(
    library.find_many_by_local_ids(
        [
            {"plex_id": "111017"},
            {"plex_id": "111018"},
            {"plex_id": "111023"},
            {"plex_id": "111024"},
        ]
    )
)  # type: ignore # TODO: use generic type in ABC
new_tracks = list(filter(None, new_tracks))
new_tracks

In [ ]:
# Add the tracks (until now the playlist is empty)
with pl.remote_edit():
    pl.tracks = new_tracks  # type: ignore # TODO

[t.id for t in pl.tracks]

In [ ]:
# Remove a track
with pl.remote_edit():
    pl.tracks.pop()

[t.id for t in pl.tracks]

In [ ]:
# Change order
with pl.remote_edit():
    pl.tracks.insert(1, pl.tracks.pop())

[t.id for t in pl.tracks]